In [ ]:

%pip install -q yfinance

import numpy as np
import pandas as pd
import yfinance as yf

print("All financial analysis libraries loaded successfully!")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)


In [ ]:

# Download historical financial prices

# Select our three ETFs
tickers = ["SPY", "GLD", "BND"]

# Download historical data
data = yf.download(
    tickers,
    start="2021-01-01",
    end="2026-10-01",
    auto_adjust=True,
    progress=False,
    threads=False
)

# Check whether data was downloaded
if data.empty:
    print("Download failed. Check your connection.")
else:
    # Extract adjusted closing prices
    prices = data["Close"][tickers].dropna()

    print("Historical Market Prices:")
    display(prices.head())

    print("Total trading days:", len(prices))


In [ ]:

# Save Financial Data

from google.colab import files

# Prepare the downloaded data
prices_clean = prices.copy()
prices_clean.columns.name = None

# Save historical prices as a CSV file
file_name = "market_prices.csv"

prices_clean.to_csv(
    file_name,
    index_label="Date"
)

print("Market data saved successfully!")
print("File name:", file_name)
print("Total records:", len(prices_clean))

# Download the CSV file to your computer
files.download(file_name)


In [ ]:

# Calculate Real Daily Returns

# Calculate daily percentage returns
daily_returns = prices.pct_change(
    fill_method=None
) * 100

# Remove the first row with missing returns
daily_returns = daily_returns.dropna()

# Display the first five rows
print("Daily Returns (%)")
display(daily_returns.head().round(2))

# Calculate average daily returns
average_returns = daily_returns.mean()

print("\nAverage Daily Returns (%)")
print(average_returns.round(4))

# Count the observations
print("\nTotal Return Records:", len(daily_returns))


In [ ]:

#Compare Real ETF Performance

import matplotlib.pyplot as plt

# Normalize all starting prices to 100
normalized_prices = (
    prices / prices.iloc[0]
) * 100

# Display first five rows
print("Normalized ETF Prices:")
display(normalized_prices.head().round(2))

# Create graph
plt.figure(figsize=(12, 6))

# Plot each ETF
plt.plot(
    normalized_prices.index,
    normalized_prices["SPY"],
    label="SPY - Stocks"
)

plt.plot(
    normalized_prices.index,
    normalized_prices["GLD"],
    label="GLD - Gold"
)

plt.plot(
    normalized_prices.index,
    normalized_prices["BND"],
    label="BND - Bonds"
)

# Add reference line
plt.axhline(100, color="gray", linestyle="--")

# Graph labels
plt.title("Historical ETF Performance Comparison")
plt.xlabel("Date")
plt.ylabel("Normalized Adjusted Price (Start = 100)")

plt.legend()
plt.grid(True, alpha=0.3)

# Display graph
plt.show()



In [ ]:

#Historical Volatility Analysis

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Calculate daily volatility
# daily_returns contains percentage returns
daily_volatility = daily_returns.std()

# Convert daily volatility to annualized volatility
annual_volatility = daily_volatility * np.sqrt(252)

# Create a comparison table
volatility_table = pd.DataFrame({
    "Daily Volatility (%)": daily_volatility,
    "Annualized Volatility (%)": annual_volatility
})

# Display results
print("Historical Volatility Analysis")
display(volatility_table.round(2))

# Identify highest and lowest volatility
highest_risk = annual_volatility.idxmax()
lowest_risk = annual_volatility.idxmin()

print("Highest Volatility Asset:", highest_risk)
print("Lowest Volatility Asset:", lowest_risk)

# Create a bar chart
plt.figure(figsize=(9, 5))

annual_volatility.plot(kind="bar")

plt.title("Annualized Historical Volatility")
plt.xlabel("ETF")
plt.ylabel("Annualized Volatility (%)")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.show()


In [ ]:

# Correlation Matrix and Heatmap

import matplotlib.pyplot as plt

# Calculate correlation between ETFs
correlation_matrix = daily_returns.corr()

# Display correlation matrix
print("Correlation Matrix:")
display(correlation_matrix.round(2))

# Create a heatmap
fig, ax = plt.subplots(figsize=(7, 5))

heatmap = ax.imshow(
    correlation_matrix,
    cmap="RdBu_r",
    vmin=-1,
    vmax=1
)

# ETF names
assets = correlation_matrix.columns

ax.set_xticks(range(len(assets)))
ax.set_yticks(range(len(assets)))

ax.set_xticklabels(assets)
ax.set_yticklabels(assets)

# Display correlation numbers inside boxes
for i in range(len(assets)):
    for j in range(len(assets)):
        value = correlation_matrix.iloc[i, j]

        ax.text(
            j, i,
            f"{value:.2f}",
            ha="center",
            va="center",
            color="black",
            bbox=dict(
                facecolor="white",
                alpha=0.8,
                edgecolor="none"
            )
        )

# Add color scale
plt.colorbar(heatmap, ax=ax, label="Correlation")

plt.title("Correlation Heatmap - SPY, GLD, BND")
plt.tight_layout()
plt.show()


In [ ]:

# Covariance Matrix and Portfolio Risk

import numpy as np

# Select our financial assets
assets = ["SPY", "GLD", "BND"]

# Calculate sample covariance matrix
covariance_matrix = daily_returns[assets].cov()

# Display covariance matrix
print("Daily Covariance Matrix:")
display(covariance_matrix.round(4))

# Create equal portfolio weights
# 1/3 of money in each asset
weights = np.array([1/3, 1/3, 1/3])

# Calculate portfolio variance
portfolio_variance = (
    weights @ covariance_matrix.to_numpy() @ weights
)

# Calculate daily portfolio volatility
portfolio_volatility = np.sqrt(portfolio_variance)

# Calculate annualized volatility
annualized_volatility = portfolio_volatility * np.sqrt(252)

# Display results
print("\nEqual-Weight Portfolio")
print("SPY Allocation: 33.33%")
print("GLD Allocation: 33.33%")
print("BND Allocation: 33.33%")

print(f"\nDaily Portfolio Volatility: {portfolio_volatility:.2f}%")
print(f"Annualized Portfolio Volatility: {annualized_volatility:.2f}%")


In [ ]:

# Real Portfolio Optimization

import numpy as np
import pandas as pd
from scipy.optimize import minimize

# Step 1: Define our assets
assets = ["SPY", "GLD", "BND"]

# Step 2: Calculate covariance matrix
cov_matrix = daily_returns[assets].cov().to_numpy()

# Step 3: Define portfolio volatility function
def portfolio_volatility(weights):

    portfolio_variance = (
        weights @ cov_matrix @ weights
    )

    volatility = np.sqrt(max(portfolio_variance, 0))

    return volatility


# Step 4: Initial equal weights
initial_weights = np.array([1/3, 1/3, 1/3])

# Step 5: Investment constraints
# Each weight must be between 0 and 1
bounds = [(0, 1), (0, 1), (0, 1)]

# Total weights must equal 1 (100%)
constraints = {
    "type": "eq",
    "fun": lambda weights: np.sum(weights) - 1
}

# Step 6: Run optimization
result = minimize(
    portfolio_volatility,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

# Check whether optimization succeeded
if not result.success:
    raise RuntimeError(result.message)

# Step 7: Get optimized weights
optimal_weights = result.x

# Step 8: Calculate portfolio risk
equal_risk = portfolio_volatility(initial_weights)
optimized_risk = portfolio_volatility(optimal_weights)

# Annualize volatility
equal_annual_risk = equal_risk * np.sqrt(252)
optimized_annual_risk = optimized_risk * np.sqrt(252)

# Step 9: Display allocation comparison
comparison = pd.DataFrame({
    "Asset": assets,
    "Equal Weight (%)": initial_weights * 100,
    "Optimized Weight (%)": optimal_weights * 100
})

print("Portfolio Allocation Comparison")
display(comparison.round(2))

# Step 10: Display risk comparison
print("\nAnnualized Portfolio Volatility")
print(f"Equal-Weight Portfolio: {equal_annual_risk:.2f}%")
print(f"Optimized Portfolio: {optimized_annual_risk:.2f}%")

# Calculate historical volatility reduction
risk_reduction = (
    (equal_annual_risk - optimized_annual_risk)
    / equal_annual_risk
) * 100

print(f"Relative Volatility Reduction: {risk_reduction:.2f}%")


In [ ]:

# Lesson 16: Visualize Portfolio Optimization Results

import matplotlib.pyplot as plt

# Graph 1: Compare portfolio allocations

plot_data = comparison.set_index("Asset")

ax = plot_data[
    ["Equal Weight (%)", "Optimized Weight (%)"]
].plot(
    kind="bar",
    figsize=(9, 5)
)

ax.set_title("Portfolio Allocation Comparison")
ax.set_xlabel("Financial Assets")
ax.set_ylabel("Portfolio Allocation (%)")

ax.set_ylim(0, 105)
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


# Graph 2: Compare portfolio volatility

portfolio_names = ["Equal-Weight", "Optimized"]

portfolio_risks = [
    equal_annual_risk,
    optimized_annual_risk
]

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    portfolio_names,
    portfolio_risks
)

ax.set_title("Portfolio Risk Comparison")
ax.set_xlabel("Portfolio Type")
ax.set_ylabel("Annualized Volatility (%)")

ax.set_ylim(0, max(portfolio_risks) * 1.25)
ax.grid(axis="y", alpha=0.3)

# Add numerical labels above each bar
for bar, risk in zip(bars, portfolio_risks):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.15,
        f"{risk:.2f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()


In [ ]:

# Lesson 17: Out-of-Sample Portfolio Testing

import numpy as np
import pandas as pd
from scipy.optimize import minimize

# Step 1: Select financial assets
assets = ["SPY", "GLD", "BND"]

# Step 2: Prepare daily return data
returns = daily_returns[assets].copy()

# Ensure dates are in correct order
returns.index = pd.to_datetime(returns.index)
returns = returns.sort_index()

# Step 3: Divide data into training and testing
train_data = returns.loc[
    "2021-01-01":"2024-12-31"
]

test_data = returns.loc[
    "2025-01-01":
]

print("Training observations:", len(train_data))
print("Testing observations:", len(test_data))

# Check both datasets
if len(train_data) < 2 or len(test_data) < 2:
    raise ValueError("Not enough training or testing data.")

# Step 4: Covariance from TRAINING data only
train_cov = train_data.cov().to_numpy()

# Step 5: Function to calculate portfolio variance
def portfolio_variance(weights):
    return weights @ train_cov @ weights

# Step 6: Initial equal weights
equal_weights = np.array([1/3, 1/3, 1/3])

# Step 7: Run optimization using training data
result = minimize(
    portfolio_variance,
    equal_weights,
    method="SLSQP",
    bounds=[(0, 1)] * 3,
    constraints={
        "type": "eq",
        "fun": lambda w: np.sum(w) - 1
    }
)

if not result.success:
    raise RuntimeError(result.message)

# Save optimized weights
train_optimal_weights = result.x

# Step 8: Function to evaluate volatility
def calculate_annual_volatility(data, weights):

    # Calculate portfolio daily returns
    portfolio_returns = data @ weights

    # Annualized standard deviation
    volatility = (
        portfolio_returns.std(ddof=1)
        * np.sqrt(252)
    )

    return volatility

# Step 9: Training period comparison
train_equal_risk = calculate_annual_volatility(
    train_data, equal_weights
)

train_optimal_risk = calculate_annual_volatility(
    train_data, train_optimal_weights
)

# Step 10: Testing period comparison
test_equal_risk = calculate_annual_volatility(
    test_data, equal_weights
)

test_optimal_risk = calculate_annual_volatility(
    test_data, train_optimal_weights
)

# Step 11: Display optimized allocation
allocation_table = pd.DataFrame({
    "Asset": assets,
    "Equal Weight (%)": equal_weights * 100,
    "Optimized Weight (%)": train_optimal_weights * 100
})

print("\nPortfolio Allocation (Training-Based)")
display(allocation_table.round(2))

# Step 12: Display risk comparison
risk_table = pd.DataFrame({
    "Period": ["Training", "Testing"],
    "Equal-Weight Volatility (%)": [
        train_equal_risk,
        test_equal_risk
    ],
    "Optimized Volatility (%)": [
        train_optimal_risk,
        test_optimal_risk
    ]
})

print("\nAnnualized Volatility Comparison")
display(risk_table.round(2))

# Step 13: Calculate out-of-sample risk reduction
test_reduction = (
    (test_equal_risk - test_optimal_risk)
    / test_equal_risk
) * 100

print(
    f"\nOut-of-Sample Volatility Reduction: "
    f"{test_reduction:.2f}%"
)


In [ ]:

# Lesson 18: Portfolio Performance Visualization

import matplotlib.pyplot as plt

# GRAPH 1: Training vs Testing Volatility

# Use the risk_table created in Lesson 17
plot_data = risk_table.set_index("Period")

ax = plot_data.plot(
    kind="bar",
    figsize=(10, 6),
    width=0.65
)

plt.title("Training vs Testing Portfolio Volatility")
plt.xlabel("Data Period")
plt.ylabel("Annualized Volatility (%)")
plt.xticks(rotation=0)

plt.grid(axis="y", alpha=0.3)
plt.legend(title="Portfolio Type")

# Display volatility above each bar
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.2f%%",
        padding=3
    )

plt.tight_layout()

# Save the first graph
plt.savefig(
    "training_testing_volatility.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()


# GRAPH 2: Portfolio Allocation Comparison

# Use the allocation_table from Lesson 17
allocation_plot = allocation_table.set_index("Asset")

ax = allocation_plot[
    ["Equal Weight (%)", "Optimized Weight (%)"]
].plot(
    kind="bar",
    figsize=(10, 6),
    width=0.65
)

plt.title("Equal-Weight vs Optimized Portfolio")
plt.xlabel("Financial Asset")
plt.ylabel("Portfolio Allocation (%)")
plt.xticks(rotation=0)

plt.ylim(0, 105)
plt.grid(axis="y", alpha=0.3)

plt.legend(title="Allocation Method")

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.2f%%",
        padding=3,
        fontsize=9
    )

plt.tight_layout()

# Save the second graph
plt.savefig(
    "portfolio_allocation_comparison.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("Both graphs generated successfully!")


In [ ]:

from google.colab import files

# Download Graph 1
files.download("training_testing_volatility.png")

# Download Graph 2
files.download("portfolio_allocation_comparison.png")


In [ ]:

# Lesson 19: Portfolio Returns and Performance

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Step 1: Convert testing returns from % to decimal
test_returns = test_data[assets] / 100

# Step 2: Calculate daily portfolio returns
equal_daily_returns = test_returns @ equal_weights

optimized_daily_returns = (
    test_returns @ train_optimal_weights
)

# Step 3: Combine portfolio returns into a table
portfolio_daily = pd.DataFrame({
    "Equal-Weight": equal_daily_returns,
    "Optimized": optimized_daily_returns
})

# Step 4: Calculate cumulative portfolio values
# Both portfolios start with a value of 100
portfolio_values = (
    (1 + portfolio_daily).cumprod() * 100
)

# Add the initial value before testing begins
initial_row = pd.DataFrame(
    {
        "Equal-Weight": [100.0],
        "Optimized": [100.0]
    },
    index=[train_data.index[-1]]
)

portfolio_values = pd.concat([
    initial_row,
    portfolio_values
])

# Step 5: Calculate total testing-period returns
final_values = portfolio_values.iloc[-1]

total_returns = (final_values / 100 - 1) * 100

# Step 6: Display comparison
comparison_table = pd.DataFrame({
    "Portfolio": ["Equal-Weight", "Optimized"],
    "Initial Value": [100, 100],
    "Final Value": [
        final_values["Equal-Weight"],
        final_values["Optimized"]
    ],
    "Total Return (%)": [
        total_returns["Equal-Weight"],
        total_returns["Optimized"]
    ]
})

print("Out-of-Sample Portfolio Performance")
display(comparison_table.round(2))

# Step 7: Plot cumulative performance
plt.figure(figsize=(12, 6))

plt.plot(
    portfolio_values.index,
    portfolio_values["Equal-Weight"],
    label="Equal-Weight Portfolio"
)

plt.plot(
    portfolio_values.index,
    portfolio_values["Optimized"],
    label="Optimized Portfolio"
)

plt.axhline(
    100,
    color="gray",
    linestyle="--",
    label="Initial Value"
)

plt.title("Out-of-Sample Cumulative Portfolio Performance")
plt.xlabel("Date")
plt.ylabel("Portfolio Value (Starting at 100)")

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save the graph
plt.savefig(
    "cumulative_portfolio_performance.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# Lesson 20: Advanced Portfolio Risk Analysis

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Daily portfolio returns from Lesson 19
# These returns are in decimal form
returns = portfolio_daily.copy()

# Assumed annual risk-free rate: 4%
risk_free_annual = 0.04

# Convert annual risk-free rate to daily rate
risk_free_daily = (
    (1 + risk_free_annual) ** (1 / 252) - 1
)

# ---------------------------------
# 1. SHARPE RATIO
# ---------------------------------

# Average daily portfolio return
mean_daily_return = returns.mean()

# Standard deviation of daily returns
daily_volatility = returns.std()

# Annualized Sharpe Ratio
sharpe_ratio = (
    (mean_daily_return - risk_free_daily)
    / daily_volatility
) * np.sqrt(252)


# ---------------------------------
# 2. MAXIMUM DRAWDOWN
# ---------------------------------

# Growth of an initial investment of 1
wealth = (1 + returns).cumprod()

# Highest portfolio value reached so far
running_peak = wealth.cummax().clip(lower=1)

# Calculate percentage drawdown
drawdown = (
    wealth / running_peak - 1
) * 100

# Largest historical drawdown
max_drawdown = drawdown.min()


# ---------------------------------
# 3. VALUE AT RISK (VaR)
# ---------------------------------

# Historical 95% one-day VaR
var_95 = -returns.quantile(0.05) * 100


# ---------------------------------
# 4. COMPARISON TABLE
# ---------------------------------

risk_metrics = pd.DataFrame({
    "Sharpe Ratio": sharpe_ratio,
    "Maximum Drawdown (%)": max_drawdown,
    "Daily VaR 95% (%)": var_95
})

print("Portfolio Risk Analysis")
display(risk_metrics.round(2))


# ---------------------------------
# 5. DRAWDOWN GRAPH
# ---------------------------------

plt.figure(figsize=(12, 6))

plt.plot(
    drawdown.index,
    drawdown["Equal-Weight"],
    label="Equal-Weight Portfolio"
)

plt.plot(
    drawdown.index,
    drawdown["Optimized"],
    label="Optimized Portfolio"
)

plt.axhline(0, color="gray", linestyle="--")

plt.title("Out-of-Sample Portfolio Drawdowns")
plt.xlabel("Date")
plt.ylabel("Drawdown (%)")

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    "portfolio_drawdown_comparison.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# Lesson 21: Constrained Portfolio Optimization

import numpy as np
import pandas as pd
from scipy.optimize import minimize

# Step 1: Select assets
assets = ["SPY", "GLD", "BND"]

# Step 2: Training covariance matrix
# Returns are expressed in percentage points
cov_matrix = train_data[assets].cov().to_numpy()

# Step 3: Define portfolio variance
def calculate_variance(weights):
    return weights @ cov_matrix @ weights

# Step 4: Starting allocation
initial_weights = np.array([1/3, 1/3, 1/3])

# Step 5: New allocation constraints
# Minimum 10%, maximum 60% in each ETF
bounds = [
    (0.10, 0.60),
    (0.10, 0.60),
    (0.10, 0.60)
]

# All weights must sum to 100%
constraints = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

# Step 6: Run constrained optimization
result = minimize(
    calculate_variance,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

if not result.success:
    raise RuntimeError(result.message)

# Step 7: Save optimized weights
constrained_weights = result.x

# Check the constraints
assert np.isclose(constrained_weights.sum(), 1)
assert np.all(constrained_weights >= 0.10 - 1e-6)
assert np.all(constrained_weights <= 0.60 + 1e-6)

# Step 8: Compare allocations
allocation_comparison = pd.DataFrame({
    "Asset": assets,
    "Equal Weight (%)": equal_weights * 100,
    "Previous Optimized (%)": train_optimal_weights * 100,
    "Constrained Optimized (%)": constrained_weights * 100
})

print("Portfolio Allocation Comparison")
display(allocation_comparison.round(2))


# Step 9: Evaluate on unseen testing data
# Convert percentage returns into decimal returns
test_decimal = test_data[assets] / 100

# Define three portfolio strategies
portfolios = {
    "Equal-Weight": equal_weights,
    "Minimum-Variance": train_optimal_weights,
    "Constrained": constrained_weights
}

results = []

for name, weights in portfolios.items():

    # Calculate daily portfolio returns
    portfolio_returns = test_decimal @ weights

    # Annualized volatility in percentage
    volatility = (
        portfolio_returns.std(ddof=1)
        * np.sqrt(252) * 100
    )

    # Total cumulative return in percentage
    total_return = (
        (1 + portfolio_returns).prod() - 1
    ) * 100

    results.append({
        "Portfolio": name,
        "Annual Volatility (%)": volatility,
        "Total Return (%)": total_return
    })

# Step 10: Display comparison
results_table = pd.DataFrame(results)

print("\nOut-of-Sample Portfolio Comparison")
display(results_table.round(2))
